# %% [markdown]
# # Step 1: Script Overview
# ------------------------
# This script automates ESP-r simulations across building folders with strict validation.
#
# Key improvements:
# - Ensures ALL 4 archetype folders exist before running simulations
# - Supports two valid archetype sets:
#     1. SemiDetached_2F_* (fab, roof, walls, windows)
#     2. Detached_2F_* (fab, roof, walls, windows)
# - Skips buildings unless a COMPLETE set is present
# - Runs simulations for ALL 4 folders in the valid set (not just the first match)
# - Dynamically selects correct .cfg file based on archetype
# - Maintains robust logging and error handling
#
# This ensures consistency and avoids partial or invalid simulation runs.

In [1]:
# %%
# Step 2: Import Libraries and Define Helper Function
# --------------------------------------------------
import os
import subprocess
from pathlib import Path

def run(cmd):
    print(f"\n>>> {cmd}")
    subprocess.run(cmd, shell=True, executable="/bin/bash")

In [2]:
# %%
# Step 3: Load ESP-r Environment
# -----------------------------
os.chdir(os.path.expanduser("~"))

run("source .profile")
run("which prj")  # confirm ESP-r is accessible


>>> source .profile

>>> which prj


In [3]:
# %%
# Step 4: Define Base Paths and Input Files
# ----------------------------------------
base_dir = Path("/Users/jakegehrung/Desktop/data_raw/baseline_models")

input_files = [
    "update_cons.txt"
]

print(f"Base directory set to: {base_dir}")

Base directory set to: /Users/jakegehrung/Desktop/data_raw/baseline_models


In [4]:
# %%
# Step 5: Identify All Building Directories
# ----------------------------------------
building_dirs = [d for d in base_dir.iterdir() if d.is_dir()]

print(f"Found {len(building_dirs)} building folders.")

Found 117 building folders.


In [5]:
# %%
# Step 6: Define Required Archetype Sets
# -------------------------------------
semi_set = {
    "SemiDetached_2F_fab",
    "SemiDetached_2F_roof",
    "SemiDetached_2F_walls",
    "SemiDetached_2F_windows",
}

detached_set = {
    "Detached_2F_fab",
    "Detached_2F_roof",
    "Detached_2F_walls",
    "Detached_2F_windows",
}

In [6]:
# %%
# Step 7: Process Buildings with Full Archetype Validation
# -------------------------------------------------------
summary_results = {}
skipped_buildings = []

for building in building_dirs:
    print(f"\n========== Processing Building: {building.name} ==========")
    
    subfolders = {d.name: d for d in building.iterdir() if d.is_dir()}
    
    # Determine which full set exists
    if semi_set.issubset(subfolders.keys()):
        selected_set = semi_set
        archetype_prefix = "SemiDetached_2F"
    elif detached_set.issubset(subfolders.keys()):
        selected_set = detached_set
        archetype_prefix = "Detached_2F"
    else:
        print("Required full set of archetype folders not found. Skipping.")
        skipped_buildings.append(building.name)
        continue
    
    print(f"Using archetype set: {archetype_prefix}")
    
    building_results = {}
    
    # Loop through ALL 4 folders in the valid set
    for folder_name in selected_set:
        archetype_dir = subfolders[folder_name]
        cfg_dir = archetype_dir / "cfg"
        
        print(f"\n--- Processing folder: {folder_name} ---")
        
        if not cfg_dir.exists():
            print("No cfg folder found. Skipping building.")
            skipped_buildings.append(building.name)
            break
        
        cfg_file = cfg_dir / "SemiDetached_2F.cfg"
        
        if not cfg_file.exists():
            print("CFG file missing. Skipping building.")
            skipped_buildings.append(building.name)
            break
        
        # Check input files
        missing_files = [f for f in input_files if not (cfg_dir / f).exists()]
        
        if missing_files:
            print(f"Missing input files: {missing_files}. Skipping building.")
            skipped_buildings.append(building.name)
            break
        
        os.chdir(cfg_dir)
        
        for input_file in input_files:
            print(f"\nRunning {input_file} in {folder_name}")
            
            cmd = f"""
            source ~/.profile
            cd "{cfg_dir}"
            prj -mode script -file SemiDetached_2F.cfg < {input_file}
            """
            
            result = subprocess.run(
                cmd,
                shell=True,
                executable="/bin/bash",
                capture_output=True,
                text=True
            )
            
            key = f"{folder_name}__{input_file}"
            building_results[key] = result
            
            print("Return Code:", result.returncode)
            print("\nSTDOUT:\n", result.stdout)
            print("\nSTDERR:\n", result.stderr)
    
    else:
        # Only save results if ALL 4 folders succeeded
        summary_results[building.name] = building_results


========== Processing Building: 1001664924 ==========
Using archetype set: SemiDetached_2F

--- Processing folder: SemiDetached_2F_windows ---

Running update_cons.txt in SemiDetached_2F_windows
Return Code: 0

STDOUT:
 Welcome to the Project manager of ESP-r V13.3.17 of September 2023.
 
 In /tmp/update_notes.txt error reading (upgrade notes) EOF sensed.
Most recent published ESP-r version is V13.3.17 of 6 March 2024
For upgrade history and download visit:
https://www.strath.ac.uk/research/energysystemsresearchunit/applications/esp-r
 
 
FAILURE: in ../ctl/SemiDetached_2F.ctl:     0    0    0    0
 the actuator 1st value (0) > allowable maximum -1!
 
FAILURE: in ../ctl/SemiDetached_2F.ctl:     0    0    0    0
 the actuator 1st value (0) > allowable maximum -1!
Model management:
  a introduction                       ..... Import & export .....     
  b databases                         n invoke CAD tool                
  c self testing                      o import CAD file         

In [7]:
# %%
# Step 8: Summary of Results
# -------------------------
failed_runs = []

for building, results in summary_results.items():
    for file, result in results.items():
        if result.returncode != 0:
            failed_runs.append((building, file))

print("\n========== SUMMARY ==========")

if failed_runs:
    print("\nFailed simulations:")
    for b, f in failed_runs:
        print(f"- Building {b}, File: {f}")
else:
    print("\nAll simulations completed successfully.")

if skipped_buildings:
    print("\nSkipped buildings:")
    for b in skipped_buildings:
        print(f"- {b}")
else:
    print("\nNo buildings were skipped.")


========== SUMMARY ==========

All simulations completed successfully.

Skipped buildings:
- 1001063639
- 1001829071
- 1234761002
- 1002539407
- 1001991633
- 1001664922
- 1234761003
- 1234761004
- 1002313096
- 1001870878
- 1002143357
- 1234647955
- 1001906269
- 1001951903
- 1002143351
- 1001627570
- 1001430744
- 1234760995
- 1001991628
- 1001664938
- 1000238925
- 1002143291
- 1001829063
- 1001951854
- 1001870864
- 1001991629
- 1001870855
- 1001664937
- 1000238923
- 1000238924
- 1234806526
- 1000336711
- 1002313093
- 1235277376
- 1234761000
- 1001627568
- 1001951906
- 1002143352
- 1002143355
- 1002143348
- 1001906271
- 1000238921
- 1001829069
- 1001063643
- 1001829066
- 1001829057
- 1002143293
- 1234760999
